In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

df = pd.read_csv('hotel_bookings.csv')

In [2]:
print(f"[Original] Total: {len(df):,} | Cancellation Rate: {df['is_canceled'].mean()*100:.1f}%")

df_clean = df.drop_duplicates()
print(f"[After Dedup] Total: {len(df_clean):,} | Cancellation Rate: {df_clean['is_canceled'].mean()*100:.1f}%")

df_clean = df_clean[df_clean['lead_time'] <= 300]
print(f"[lead_time ≤ 300] Total: {len(df_clean):,} | Cancellation Rate: {df_clean['is_canceled'].mean()*100:.1f}%")

df_sample, _ = train_test_split(df_clean, train_size=30000, stratify=df_clean['is_canceled'], random_state=42)
print(f"[Stratified Sample] Total: {len(df_sample):,} | Cancellation Rate: {df_sample['is_canceled'].mean()*100:.1f}%")

steps = [
    (df, 'Original\n(119,390)'),
    (df_clean, 'Dedup + Filter\n(85,110)'),
    (df_sample, 'Stratified Sample\n(30,000)')
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = [sns.color_palette('muted')[0], sns.color_palette('muted')[3]]

for ax, (data, title) in zip(axes, steps):
    counts = data['is_canceled'].value_counts().sort_index()
    bars = ax.bar(['Not Canceled', 'Canceled'], counts.values, color=colors, width=0.4, edgecolor='white')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Count')
    ax.set_ylim(0, counts.max() * 1.15)
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + counts.max()*0.02,
                f'{v:,}', ha='center', fontsize=10, fontweight='bold')
    sns.despine(ax=ax)

plt.suptitle('Data Preprocessing Steps', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

[Original] Total: 119,390 | Cancellation Rate: 37.0%
[After Dedup] Total: 87,396 | Cancellation Rate: 27.5%
[lead_time ≤ 300] Total: 85,110 | Cancellation Rate: 27.0%
[Stratified Sample] Total: 30,000 | Cancellation Rate: 27.0%


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_clean['lead_time'], bins=60, ax=axes[0], color=sns.color_palette('muted')[0], edgecolor='white')
axes[0].set_title('Lead Time Distribution (Before Sampling)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Lead Time (days)')
axes[0].set_ylabel('Count')
sns.despine(ax=axes[0])

sns.histplot(df_sample['lead_time'], bins=60, ax=axes[1], color=sns.color_palette('muted')[2], edgecolor='white')
axes[1].set_title('Lead Time Distribution (After Sampling)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Lead Time (days)')
axes[1].set_ylabel('Count')
sns.despine(ax=axes[1])

plt.suptitle('Lead Time Distribution: Before vs. After Stratified Sampling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Feature Engineering

## Removed Features
| Feature | Reason |
|---------|--------|
| `arrival_date_day_of_month` | Noise — no meaningful pattern with cancellation |
| `arrival_date_year` | Only 3 unique values; negligible variation in cancellation rate |
| `babies` | Extremely sparse data |
| `company` | Over 94% missing values |
| `reservation_status` / `reservation_status_date` | Data leakage — directly encodes cancellation outcome |
| `deposit_type` | Non Refund shows 99.4% cancellation rate artifact; Refundable has too few records |
| `room_changed` | Data leakage — room assignment happens at check-in, so canceled guests always show no change |
| `is_repeated_guest` | Correlation 0.42 with `previous_bookings_not_canceled`; latter has stronger signal |
| `required_car_parking_spaces` | Suspected leakage — parking spaces assigned at check-in show 0% cancellation |

## Engineered Features
| Feature | Description |
|---------|-------------|
| `has_children` | 1 if children > 0 or babies > 0 |
| `is_domestic` | 1 if country is Portugal (PRT) |
| `has_prev_completed` | 1 if previous_bookings_not_canceled > 0 |
| `has_booking_changes` | 1 if booking_changes > 0 |
| `lead_x_domestic` | Interaction term: lead_time × is_domestic |
| `total_nights` | Sum of weekend and weekday nights |
| `country_group` | Top 10 countries + Others |
| `agent` | Restored: NaN filled with 0 (direct booking) |

## Encoding
| Feature | Method |
|---------|--------|
| `hotel` | Label Encoding (City Hotel=0, Resort Hotel=1) |
| `arrival_date_month` | Integer mapping (1–12) |
| `market_segment` | Label Encoding |
| `customer_type` | Label Encoding |

## Scaling
| Method | Applied To |
|--------|-----------|
| `StandardScaler` | All numerical features before model training |

In [ ]:
df_model = df_sample.copy()

# Engineer new features
df_model['has_children'] = ((df_model['children'] > 0) | (df_model['babies'] > 0)).astype(int)
df_model['is_domestic'] = (df_model['country'] == 'PRT').astype(int)
df_model['has_prev_completed'] = (df_model['previous_bookings_not_canceled'] > 0).astype(int)
df_model['has_booking_changes'] = (df_model['booking_changes'] > 0).astype(int)
df_model['lead_x_domestic'] = df_model['lead_time'] * df_model['is_domestic']
df_model['total_nights'] = df_model['stays_in_weekend_nights'] + df_model['stays_in_week_nights']

# Month to integer mapping (1–12)
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}
df_model['arrival_date_month'] = df_model['arrival_date_month'].map(month_map)

# Label Encoding for categorical features
hotel_map = {'City Hotel': 0, 'Resort Hotel': 1}
df_model['hotel'] = df_model['hotel'].map(hotel_map)

market_map = {
    'Online TA': 0, 'Offline TA/TO': 1, 'Direct': 2,
    'Corporate': 3, 'Groups': 4, 'Complementary': 5, 'Aviation': 6
}
df_model['market_segment'] = df_model['market_segment'].map(market_map)

customer_map = {
    'Transient': 0, 'Transient-Party': 1, 'Contract': 2, 'Group': 3
}
df_model['customer_type'] = df_model['customer_type'].map(customer_map)

# Visualization-only segments (excluded from model training)
df_model['lead_segment'] = pd.cut(
    df_model['lead_time'],
    bins=[-1, 14, 90, 300],
    labels=['Impulse (0–14d)', 'Regular (15–90d)', 'Planned (91–300d)']
)
df_model['domestic_segment'] = df_model['is_domestic'].map({1: 'Domestic (PRT)', 0: 'International'})
df_model['season_segment'] = df_model['arrival_date_month'].apply(
    lambda x: 'Peak (Jun–Sep)' if x in [6, 7, 8, 9] else 'Off-Peak'
)
df_model['price_segment'] = pd.cut(
    df_model['adr'],
    bins=[-1, 72, 134, 500],
    labels=['Low (~72)', 'Mid (72–134)', 'High (134~)']
)

# Final feature set
features = [
    'lead_time', 'hotel', 'total_of_special_requests',
    'is_domestic', 'stays_in_weekend_nights', 'stays_in_week_nights',
    'arrival_date_month', 'arrival_date_week_number', 'market_segment',
    'has_children', 'previous_cancellations', 'adr', 'customer_type',
    'has_prev_completed', 'has_booking_changes',
    'lead_x_domestic', 'total_nights',
]

X = df_model[features]
y = df_model['is_canceled']

print(f"Number of features: {len(features)}")
print(f"Number of samples : {len(X)}")
print(X.dtypes)

# Model Development & Selection

## Domain Consideration
In the hotel industry, a **False Positive** — predicting cancellation for a guest who actually shows up — carries significant operational risk:
misallocating their room or triggering unnecessary retention efforts damages guest experience and hotel reputation.
Therefore, **Precision** was prioritized as the primary metric over F1 or Recall.
To embed this into the training process, we used **Fbeta(β=0.5)** as the scoring function, which weights Precision twice as heavily as Recall.

---

## Model Selection Process

| Step | Approach | Key Decision |
|------|----------|-------------|
| 1 | RF vs. XGBoost baseline | Both similar; RF slightly higher F1 → RF selected as base |
| 2 | Add `class_weight='balanced'` | 27%:73% class imbalance corrected; F1 0.610 → 0.634 |
| 3 | Switch scoring to Fbeta(β=0.5) | Precision-focused tuning aligned with domain priority |
| 4 | RandomizedSearchCV (50 iterations, 5-fold CV) | Hyperparameter search with Fbeta scoring |
| 5 | Threshold optimization (0.4 → 0.5) | Precision 0.609 → 0.724; acceptable Recall trade-off |
| 6 | Feature ablation: all features vs. selected | Full-feature model revealed `country` (rank 2) and `agent` (rank 4) were excluded but important |
| 7 | Restore `agent` (NaN→0) and `country_group` (top 10 + Other) | Precision improved from 0.666 to 0.724; ROC-AUC 0.855 → 0.879 |

---

## Final Model
**Random Forest** | `class_weight='balanced'` | `n_estimators=300` | `threshold=0.5`

| Metric | Not Canceled | Canceled |
|--------|-------------|---------|
| Precision | 0.89 | **0.724** |
| Recall | 0.81 | 0.568 |
| F1-score | 0.85 | 0.637 |
| **ROC-AUC** | — | **0.879** |

> At threshold=0.5, **72.4% of predicted cancellations are genuine cancellations**,
> minimizing the risk of misclassifying real guests as cancellations.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# Train/Test Split (80/20, stratified by cancellation rate)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Apply StandardScaler to normalize feature ranges
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {len(X_train):,} samples / Test: {len(X_test):,} samples")
print(f"Train cancellation rate: {y_train.mean()*100:.1f}% / Test: {y_test.mean()*100:.1f}%")
print(f"\n[Before Scaling] lead_time — mean: {X_train['lead_time'].mean():.2f}, std: {X_train['lead_time'].std():.2f}")
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
print(f"[After Scaling]  lead_time — mean: {X_train_scaled_df['lead_time'].mean():.2f}, std: {X_train_scaled_df['lead_time'].std():.2f}")

In [ ]:
# Baseline comparison (with StandardScaler applied)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)
rf_proba = rf.predict_proba(X_test_scaled)[:, 1]

xgb = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb.fit(X_train_scaled, y_train)
xgb_pred = xgb.predict(X_test_scaled)
xgb_proba = xgb.predict_proba(X_test_scaled)[:, 1]

print("=" * 40)
print("Random Forest Baseline")
print(f"  F1-score : {f1_score(y_test, rf_pred):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(y_test, rf_proba):.4f}")
print("=" * 40)
print("XGBoost Baseline")
print(f"  F1-score : {f1_score(y_test, xgb_pred):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(y_test, xgb_proba):.4f}")
print("=" * 40)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, fbeta_score

# Fbeta(beta=0.5): weights Precision twice as heavily as Recall
precision_scorer = make_scorer(fbeta_score, beta=0.5)

param_dist = {
    'n_estimators': range(100, 600, 50),
    'max_depth': range(5, 35, 5),
    'min_samples_split': range(2, 12, 2),
    'min_samples_leaf': range(1, 6),
}

rs = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    param_dist,
    n_iter=50,
    cv=5,
    scoring=precision_scorer,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

rs.fit(X_train_scaled, y_train)

print(f"Best Parameters: {rs.best_params_}")
print(f"CV Fbeta(0.5)-score: {rs.best_score_:.4f}")

In [8]:
from sklearn.metrics import precision_score, recall_score

final_model = rs.best_estimator_
final_proba = final_model.predict_proba(X_test_scaled)[:, 1]

print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print("-" * 48)
for t in [0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]:
    pred_t = (final_proba >= t).astype(int)
    p = precision_score(y_test, pred_t)
    r = recall_score(y_test, pred_t)
    f = f1_score(y_test, pred_t)
    print(f"{t:<12.2f} {p:<12.3f} {r:<12.3f} {f:<12.3f}")

Threshold    Precision    Recall       F1          
------------------------------------------------
0.40         0.609        0.704        0.653       
0.45         0.639        0.637        0.638       
0.50         0.670        0.573        0.618       
0.55         0.693        0.513        0.589       
0.60         0.718        0.454        0.556       
0.65         0.744        0.395        0.516       
0.70         0.760        0.330        0.460       


In [9]:
from sklearn.metrics import ConfusionMatrixDisplay

THRESHOLD = 0.5
final_pred = (final_proba >= THRESHOLD).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, final_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Canceled', 'Canceled'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].grid(False)
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')

# Canceled 클래스 지표만 표시
report = classification_report(y_test, final_pred, target_names=['Not Canceled', 'Canceled'], output_dict=True)
metrics = {'Precision': report['Canceled']['precision'],
           'Recall': report['Canceled']['recall'],
           'F1-score': report['Canceled']['f1-score']}

bars = axes[1].bar(metrics.keys(), metrics.values(),
                   color=sns.color_palette('muted')[3], width=0.4, edgecolor='white')
for bar, v in zip(bars, metrics.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.3f}',
                 ha='center', va='bottom', fontsize=11, fontweight='bold')

axes[1].set_ylim(0, 1.1)
axes[1].set_title('Canceled Class Performance', fontsize=13, fontweight='bold')
axes[1].xaxis.grid(False)
sns.despine(ax=axes[1])

plt.suptitle(
    f'Final Model Performance  |  Threshold={THRESHOLD}  |  ROC-AUC: {roc_auc_score(y_test, final_proba):.4f}',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [25]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# 5-Fold Cross Validation (최종 모델 평가)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_f1 = cross_val_score(final_model, X_train_scaled, y_train, cv=skf, scoring='f1')
cv_auc = cross_val_score(final_model, X_train_scaled, y_train, cv=skf, scoring='roc_auc')

print("=== 5-Fold Cross Validation Results (Final Model) ===")
print(f"F1-score  : {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")
print(f"ROC-AUC   : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
print(f"\nFold-wise F1 : {[round(s, 4) for s in cv_f1]}")
print(f"Fold-wise AUC: {[round(s, 4) for s in cv_auc]}")

=== 5-Fold Cross Validation Results (Final Model) ===
F1-score  : 0.6168 ± 0.0115
ROC-AUC   : 0.8532 ± 0.0046

Fold-wise F1 : [np.float64(0.6047), np.float64(0.6202), np.float64(0.6375), np.float64(0.6091), np.float64(0.6125)]
Fold-wise AUC: [np.float64(0.8556), np.float64(0.8552), np.float64(0.8565), np.float64(0.844), np.float64(0.8548)]


In [26]:
from sklearn.preprocessing import LabelEncoder

# 데이터 누수 컬럼만 제거
drop_cols = ['is_canceled', 'reservation_status', 'reservation_status_date']
df_all = df_sample.drop(columns=drop_cols).copy()

# 모든 str 컬럼 라벨 인코딩
for col in df_all.select_dtypes(include='object').columns:
    df_all[col] = LabelEncoder().fit_transform(df_all[col].astype(str))

X_all = df_all
y_all = df_sample['is_canceled']

X_all_train, X_all_test, y_all_train, y_all_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42
)

rf_all = RandomForestClassifier(class_weight='balanced', n_estimators=300, random_state=42)
rf_all.fit(X_all_train, y_all_train)
proba_all = rf_all.predict_proba(X_all_test)[:, 1]

print(f"피처 수: {X_all.shape[1]}")
print(f"\n{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print("-" * 48)
for t in [0.4, 0.45, 0.5, 0.55, 0.6, 0.65]:
    pred_t = (proba_all >= t).astype(int)
    p = precision_score(y_all_test, pred_t)
    r = recall_score(y_all_test, pred_t)
    f = f1_score(y_all_test, pred_t)
    print(f"{t:<12.2f} {p:<12.3f} {r:<12.3f} {f:<12.3f}")

print(f"\nROC-AUC: {roc_auc_score(y_all_test, proba_all):.4f}")

/var/folders/4t/p5h5r7_j5xq5jzrvy345s8yc0000gn/T/ipykernel_22070/1484502494.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_all.select_dtypes(include='object').columns:


피처 수: 29

Threshold    Precision    Recall       F1          
------------------------------------------------
0.40         0.701        0.715        0.708       
0.45         0.726        0.664        0.694       
0.50         0.755        0.603        0.670       
0.55         0.788        0.547        0.646       
0.60         0.812        0.485        0.607       
0.65         0.841        0.403        0.545       

ROC-AUC: 0.8975


# Feature Engineering Iteration

## Limitation of Initial Feature Selection
Training with the full feature set achieved Precision 0.755 vs. 0.666 with the selected features.
Feature importance analysis revealed two critical features that had been excluded:

| Feature | Importance Rank | Issue |
|---------|----------------|-------|
| `country` | 2nd (0.115) | Binarized to `is_domestic`, losing country-level patterns |
| `agent` | 4th (0.071) | Dropped due to missing values, but actually important |

## Improvement
- `agent` → Restored by filling NaN with 0 (represents direct bookings without an agent)
- `country` → Grouped into top 10 countries + Others (resolves high-cardinality issue)

In [ ]:
df_model2 = df_sample.copy()

# Engineer features (same as v1 + restored agent and country_group)
df_model2['has_children'] = ((df_model2['children'] > 0) | (df_model2['babies'] > 0)).astype(int)
df_model2['is_domestic'] = (df_model2['country'] == 'PRT').astype(int)
df_model2['has_prev_completed'] = (df_model2['previous_bookings_not_canceled'] > 0).astype(int)
df_model2['has_booking_changes'] = (df_model2['booking_changes'] > 0).astype(int)
df_model2['lead_x_domestic'] = df_model2['lead_time'] * df_model2['is_domestic']
df_model2['total_nights'] = df_model2['stays_in_weekend_nights'] + df_model2['stays_in_week_nights']

# Restore agent: fill NaN with 0 (direct booking = no agent)
df_model2['agent'] = df_model2['agent'].fillna(0)

# Restore country: group top 10 countries + Others
top10 = df_model2['country'].value_counts().head(10).index
df_model2['country_group'] = df_model2['country'].apply(lambda x: x if x in top10 else 'Other')
country_map = {c: i for i, c in enumerate(df_model2['country_group'].unique())}
df_model2['country_group'] = df_model2['country_group'].map(country_map)

# Month to integer mapping
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}
df_model2['arrival_date_month'] = df_model2['arrival_date_month'].map(month_map)

# Label Encoding
df_model2['hotel'] = df_model2['hotel'].map({'City Hotel': 0, 'Resort Hotel': 1})
df_model2['market_segment'] = df_model2['market_segment'].map({
    'Online TA': 0, 'Offline TA/TO': 1, 'Direct': 2,
    'Corporate': 3, 'Groups': 4, 'Complementary': 5, 'Aviation': 6
})
df_model2['customer_type'] = df_model2['customer_type'].map({
    'Transient': 0, 'Transient-Party': 1, 'Contract': 2, 'Group': 3
})

# Visualization-only segments
df_model2['lead_segment'] = pd.cut(
    df_model2['lead_time'],
    bins=[-1, 14, 90, 300],
    labels=['Impulse (0–14d)', 'Regular (15–90d)', 'Planned (91–300d)']
)
df_model2['domestic_segment'] = df_model2['is_domestic'].map({1: 'Domestic (PRT)', 0: 'International'})
df_model2['season_segment'] = df_model2['arrival_date_month'].apply(
    lambda x: 'Peak (Jun–Sep)' if x in [6, 7, 8, 9] else 'Off-Peak'
)
df_model2['price_segment'] = pd.cut(
    df_model2['adr'],
    bins=[-1, 72, 134, 500],
    labels=['Low (~72)', 'Mid (72–134)', 'High (134~)']
)

# Final feature set (v2: with agent and country_group)
features2 = [
    'lead_time', 'hotel', 'total_of_special_requests',
    'is_domestic', 'stays_in_weekend_nights', 'stays_in_week_nights',
    'arrival_date_month', 'arrival_date_week_number', 'market_segment',
    'has_children', 'previous_cancellations', 'adr', 'customer_type',
    'has_prev_completed', 'has_booking_changes', 'lead_x_domestic', 'total_nights',
    'agent', 'country_group',
]

X2 = df_model2[features2]
y2 = df_model2['is_canceled']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, stratify=y2, random_state=42
)

rf2 = RandomForestClassifier(class_weight='balanced', n_estimators=300, random_state=42)
rf2.fit(X2_train, y2_train)
proba2 = rf2.predict_proba(X2_test)[:, 1]

print(f"Number of features: {len(features2)}")
print(f"\n{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print("-" * 48)
for t in [0.4, 0.45, 0.5, 0.55, 0.6, 0.65]:
    pred_t = (proba2 >= t).astype(int)
    p = precision_score(y2_test, pred_t)
    r = recall_score(y2_test, pred_t)
    f = f1_score(y2_test, pred_t)
    print(f"{t:<12.2f} {p:<12.3f} {r:<12.3f} {f:<12.3f}")

print(f"\nROC-AUC: {roc_auc_score(y2_test, proba2):.4f}")

# 최종 모델 분석

In [12]:
domestic = df_sample[df_sample['country'] == 'PRT']
international = df_sample[df_sample['country'] != 'PRT']

print(f"Domestic (PRT)   - Count: {len(domestic):,}, Cancellation Rate: {domestic['is_canceled'].mean()*100:.1f}%")
print(f"International    - Count: {len(international):,}, Cancellation Rate: {international['is_canceled'].mean()*100:.1f}%")

Domestic (PRT)   - Count: 9,347, Cancellation Rate: 34.6%
International    - Count: 20,653, Cancellation Rate: 23.6%


In [13]:
feat_imp = pd.Series(rf2.feature_importances_, index=features2).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors = [sns.color_palette('muted')[0] if v < feat_imp.quantile(0.7)
          else sns.color_palette('muted')[3] for v in feat_imp.values]

bars = ax.barh(feat_imp.index, feat_imp.values, color=colors, edgecolor='white')
for bar, v in zip(bars, feat_imp.values):
    ax.text(v + 0.001, bar.get_y() + bar.get_height()/2,
            f'{v:.3f}', va='center', fontsize=9)

ax.set_xlabel('Feature Importance', fontsize=11)
ax.set_title('Feature Importance (Random Forest)', fontsize=13, fontweight='bold')
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

In [14]:
THRESHOLD = 0.5
final_pred2 = (proba2 >= THRESHOLD).astype(int)

segments = ['lead_segment', 'domestic_segment', 'season_segment', 'price_segment']
titles = ['Lead Time Segment', 'Domestic vs. International', 'Season', 'Price Tier']
xlabels = [
    ['Impulse\n(0–14d)', 'Regular\n(15–90d)', 'Planned\n(91–300d)'],
    ['Domestic (PRT)', 'International'],
    ['Off-Peak', 'Peak (Jun–Sep)'],
    ['Low\n(~72)', 'Mid\n(72–134)', 'High\n(134~)']
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
palette = sns.color_palette('muted')

for ax, seg, title, xlbl in zip(axes.flatten(), segments, titles, xlabels):
    seg_data = df_model2.loc[X2_test.index].copy()
    seg_data['actual'] = y2_test.values
    seg_data['predicted'] = final_pred2

    actual_rate = seg_data.groupby(seg)['actual'].mean() * 100
    pred_rate = seg_data.groupby(seg)['predicted'].mean() * 100

    x = np.arange(len(actual_rate))
    w = 0.35
    b1 = ax.bar(x - w/2, actual_rate.values, width=w, label='Actual', color=palette[0], edgecolor='white')
    b2 = ax.bar(x + w/2, pred_rate.values, width=w, label='Predicted', color=palette[3], edgecolor='white')

    for bar, v in zip(b1, actual_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)
    for bar, v in zip(b2, pred_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(xlbl)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Cancellation Rate (%)')
    ax.legend()
    sns.despine(ax=ax)

plt.suptitle('Actual vs. Predicted Cancellation Rate by Segment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Customer Clustering (K-Means)

K-Means clustering is applied to discover natural customer segments based on booking behavior.
Clusters are visualized using PCA (2D) and analyzed by cancellation rate.

In [23]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler as SS

# lead_time, is_domestic 중심 피처 (취소율과 가장 관련 높은 피처)
cluster_features = ['lead_time', 'is_domestic', 'adr', 'total_of_special_requests']
X_cluster = df_model2[cluster_features].copy()

# 스케일링 (군집화는 거리 기반이라 필수)
cluster_scaler = SS()
X_cluster_scaled = cluster_scaler.fit_transform(X_cluster)

# Elbow Method
inertia = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster_scaled)
    inertia.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_range, inertia, marker='o', color=sns.color_palette('muted')[0], linewidth=2)
ax.set_title('Elbow Method – Optimal Number of Clusters', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Inertia')
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

In [24]:
K = 2
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df_model2['cluster'] = kmeans.fit_predict(X_cluster_scaled)

cancel_by_cluster = df_model2.groupby('cluster')['is_canceled'].mean() * 100
sorted_clusters = cancel_by_cluster.sort_values(ascending=False)
cluster_labels = {sorted_clusters.index[0]: 'High-Risk', sorted_clusters.index[1]: 'Low-Risk'}
df_model2['cluster_label'] = df_model2['cluster'].map(cluster_labels)

# PCA 2D 시각화
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cluster_scaled)

label_order = ['High-Risk', 'Low-Risk']
palette = {'High-Risk': sns.color_palette('muted')[3],
           'Low-Risk': sns.color_palette('muted')[0]}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for label in label_order:
    mask = df_model2['cluster_label'] == label
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=[palette[label]], label=label, alpha=0.4, s=10)
axes[0].set_title(f'Customer Clusters (K={K}) – PCA 2D', fontsize=13, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[0].legend()
sns.despine(ax=axes[0])

cancel_by_label = df_model2.groupby('cluster_label')['is_canceled'].mean() * 100
cancel_by_label = cancel_by_label.reindex(label_order)
bars = axes[1].bar(
    cancel_by_label.index,
    cancel_by_label.values,
    color=[palette[l] for l in label_order],
    width=0.4, edgecolor='white'
)
for bar, v in zip(bars, cancel_by_label.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.5,
                 f'{v:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('Cancellation Rate by Cluster', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Cancellation Rate (%)')
axes[1].set_ylim(0, 50)
axes[1].xaxis.grid(False)
sns.despine(ax=axes[1])

plt.suptitle('K-Means Clustering: Cancellation is Not Easily Separable', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("=== Cluster Profile ===")
profile = df_model2.groupby('cluster_label')[cluster_features + ['is_canceled']].mean().round(2)
profile['is_canceled'] = (profile['is_canceled'] * 100).round(1)
profile.columns = cluster_features + ['cancel_rate (%)']
print(profile.reindex(label_order))
print("\n→ Even with 2 clusters, cancellation rates are not drastically different.")
print("  This confirms that cancellation behavior is driven by multiple interacting factors,")
print("  justifying the need for a multi-feature classification model.")

=== Cluster Profile ===
               lead_time  is_domestic     adr  total_of_special_requests  \
cluster_label                                                              
High-Risk          56.96          1.0   96.32                       0.60   
Low-Risk           79.30          0.0  111.47                       0.75   

               cancel_rate (%)  
cluster_label                   
High-Risk                 35.0  
Low-Risk                  24.0  

→ Even with 2 clusters, cancellation rates are not drastically different.
  This confirms that cancellation behavior is driven by multiple interacting factors,
  justifying the need for a multi-feature classification model.
